In [1]:
# Imports
import os, re, pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
from gaussiannb_model import GaussianNBTF, save_model

I0000 00:00:1774484330.051282   17722 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774484330.511642   17722 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774484332.100124   17722 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Configuration
data_root = "/home/prasndhdl/Downloads/avatar/Avatar-PrashantD_Forked/Data_clean"
skip_dirs = { 'Group1-8channels' }
use_pca = True
pca_components = 32  # set None to disable or adjust
var_smoothing = 1e-9  # GNB variance smoothing

In [3]:
# Load all EEG files recursively into a list of DataFrames
core_dir = pathlib.Path(data_root)
dfs = []
for item in core_dir.rglob('*.txt'):
    try:
        if set(item.parts).isdisjoint(skip_dirs):
            df = pd.read_csv(item, sep=',', header=4, on_bad_lines='skip')
            df['src_filename'] = str(item)
            dfs.append(df)
    except Exception as e:
        print(f'Failed to read {item}: {e}')
print(f'Read {len(dfs)} files')
eeg_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(eeg_data.shape)

Failed to read /home/prasndhdl/Downloads/avatar/Avatar-PrashantD_Forked/Data_clean/G04/person09/Backward/OpenBCI-RAW-2025-03-14_15-56-17.txt: No columns to parse from file
Read 1826 files
(2885310, 34)


/tmp/ipykernel_17722/290652960.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  eeg_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


In [4]:
# Robust label normalization from filenames
assert not eeg_data.empty, 'No data loaded. Check data_root.'
src = eeg_data['src_filename'].astype(str).str.lower()
src_norm = src.str.replace(r'[\s_\-]+', '', regex=True)
labels = pd.Series('', index=eeg_data.index)
def assign_where(patterns, value, labels, src_norm):
    m = pd.Series(False, index=src_norm.index)
    for pat in patterns:
        m = m | src_norm.str.contains(pat)
    cond = (labels == '') & m
    return labels.where(~cond, other=value)
labels = assign_where(['backward','backwards'], 'backward', labels, src_norm)
labels = assign_where(['fowward','forward'], 'forward', labels, src_norm)
labels = assign_where(['landing'], 'landing', labels, src_norm)
labels = assign_where(['left'], 'left', labels, src_norm)
labels = assign_where(['right'], 'right', labels, src_norm)
labels = assign_where(['takeoff','takeoff'], 'takeoff', labels, src_norm)
eeg_data['label_txt'] = labels.astype(str)
print(eeg_data.groupby(['label_txt'])['src_filename'].count())

label_txt
             55615
backward    449906
forward     472646
landing     479132
left        473995
right       481853
takeoff     472163
Name: src_filename, dtype: int64


In [5]:
# Build feature matrix X and label vector y
df = eeg_data.copy()
df = df[df['label_txt'].astype(str).str.len() > 0]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
drop_cols = [c for c in ['Sample Index'] if c in num_cols]
feature_cols = [c for c in num_cols if c not in drop_cols]
feat = df[feature_cols].copy()
feat = feat.replace([np.inf, -np.inf], np.nan)
feat = feat.dropna(axis=1, how='all')
med = feat.median(numeric_only=True)
feat = feat.fillna(med)
std = feat.std(numeric_only=True)
keep = std[std > 0].index.tolist()
feat = feat[keep]
X = feat.to_numpy(dtype=np.float32)
y_cat = df['label_txt'].astype('category')
y = y_cat.cat.codes.to_numpy(dtype=np.int32)
label_names = list(y_cat.cat.categories)
print('X shape:', X.shape, 'classes:', len(label_names))

X shape: (2829695, 27) classes: 6


In [6]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# z-score standardization (fit on train only)
mu = X_train.mean(axis=0)
sd = X_train.std(axis=0)
sd[sd == 0] = 1.0
X_train_z = (X_train - mu) / sd
X_test_z  = (X_test  - mu) / sd
# Optional PCA
pca_components_ = None
pca_mean_ = None
if use_pca and pca_components is not None:
    n_comp = int(min(pca_components, X_train_z.shape[1]))
    pca = PCA(n_components=n_comp, whiten=True, random_state=42)
    X_train_z = pca.fit_transform(X_train_z)
    X_test_z = pca.transform(X_test_z)
    pca_components_ = pca.components_.astype(np.float32)
    pca_mean_ = pca.mean_.astype(np.float32)
print('Train/Test shapes:', X_train_z.shape, X_test_z.shape)

Train/Test shapes: (2263756, 27) (565939, 27)


In [7]:
# Train Gaussian Naive Bayes (TensorFlow)
model = GaussianNBTF(var_smoothing=var_smoothing)
model.fit(X_train_z, y_train)
y_pred = model.predict(X_test_z)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

W0000 00:00:1774484397.975678   17722 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Accuracy: 0.2179
Confusion matrix:
 [[13506  8092  2862  8409 17294 39818]
 [12676 20463  2825  4471  8629 45465]
 [ 5552  8060  3203  9640 16271 53100]
 [10716  8905  2814 15044 13486 43834]
 [12166 10649  2970  8504 15703 46379]
 [ 9064 12125    32  7587 10252 55373]]

Classification report:
               precision    recall  f1-score   support

    backward       0.21      0.15      0.18     89981
     forward       0.30      0.22      0.25     94529
     landing       0.22      0.03      0.06     95826
        left       0.28      0.16      0.20     94799
       right       0.19      0.16      0.18     96371
     takeoff       0.19      0.59      0.29     94433

    accuracy                           0.22    565939
   macro avg       0.23      0.22      0.19    565939
weighted avg       0.23      0.22      0.19    565939



In [ ]:
# Save trained model with preprocessing metadata
meta = {
    'mu': mu.astype(np.float32),
    'sd': sd.astype(np.float32),
    'label_names': label_names,  # list[str]
    'feature_cols': feature_cols,  # list[str]
    'kept_feature_cols': keep,  # list[str]
    'pca_components': pca_components_ if pca_components_ is not None else None,
    'pca_mean': pca_mean_ if pca_mean_ is not None else None,
}
out_path = 'gaussiannb_trained.pth'
save_model(out_path, model, meta)
print(f'Saved trained model to {out_path}')

Saved trained model to gaussiannb_trained.pth


In [ ]:
# Reload model module to ensure latest save_model is in scope
import importlib, gaussiannb_model
importlib.reload(gaussiannb_model)
from gaussiannb_model import GaussianNBTF, save_model